# 🔐 UEBA Lab — Notebook 01: Data Ingestion & Feature Engineering

**Dataset:** CMU CERT Insider Threat Dataset r6.2  
**Goal:** Load all event logs, explore behavioral patterns, and build a 40-feature behavioral fingerprint matrix (users × days × features).

## 📋 Notebook Sections
| Section | Topic |
|---------|-------|
| 1.0 | Setup & Configuration |
| 1.1 | Load CERT r6.2 CSVs |
| 1.2 | Dataset Overview |
| 1.3 | Exploratory Data Analysis |
| 1.4 | Feature Engineering |
| 1.5 | Save Feature Matrix |

> **⚠️ Before running:** Download CERT r6.2 from  
> https://www.kaggle.com/datasets/nitishabharathi/cert-insider-threat  
> Extract to `../data/cert-insider-threat-r6.2/`

---
## 1.0 Setup & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Configuration ─────────────────────────────────────────────────────────────
CERT_DATA_PATH = '../data/cert-insider-threat-r6.2/'

SENSITIVE_KEYWORDS = [
    'model', 'proprietary', 'confidential', 'secret',
    'patent', 'source', 'algorithm', 'key', 'credential',
    'private', 'internal', 'classified', 'restricted'
]

JOB_SITE_DOMAINS = [
    'linkedin.com', 'indeed.com', 'glassdoor.com',
    'monster.com', 'careerbuilder.com', 'ziprecruiter.com'
]

CLOUD_STORAGE_DOMAINS = [
    'dropbox.com', 'drive.google.com', 'onedrive.live.com',
    'box.com', 'mega.nz', 'wetransfer.com'
]

WEBMAIL_DOMAINS = [
    'gmail.com', 'yahoo.com', 'hotmail.com',
    'outlook.com', 'protonmail.com'
]

BASELINE_WINDOW_DAYS = 30

print('✅ Configuration loaded.')
print(f'   CERT data path : {CERT_DATA_PATH}')
print(f'   Baseline window: {BASELINE_WINDOW_DAYS} days')

---
## 1.1 Load CERT r6.2 CSVs

We load all five event log files plus the LDAP organisational data.  
Each file shares a common `user` column that acts as the join key.

In [ ]:
def load_cert_csv(filename: str, date_cols: list) -> pd.DataFrame:
    """
    Load a CERT CSV file with robust error handling.
    Parses date columns and standardises the user column name.
    """
    filepath = os.path.join(CERT_DATA_PATH, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f'❌ File not found: {filepath}\n'
            f'   Please download CERT r6.2 from Kaggle and extract to:\n'
            f'   {CERT_DATA_PATH}'
        )
    df = pd.read_csv(filepath, parse_dates=date_cols, low_memory=False)
    if 'user' in df.columns:
        df.rename(columns={'user': 'user_id'}, inplace=True)
    print(f'   ✅ {filename:<25} {len(df):>10,} rows  |  cols: {list(df.columns)}')
    return df


print('📂 Loading CERT r6.2 event logs...\n')

logon  = load_cert_csv('logon.csv',  date_cols=['date'])
device = load_cert_csv('device.csv', date_cols=['date'])
files  = load_cert_csv('file.csv',   date_cols=['date'])
email  = load_cert_csv('email.csv',  date_cols=['date'])
http   = load_cert_csv('http.csv',   date_cols=['date'])

print(f'\n📊 Total events loaded: {len(logon)+len(device)+len(files)+len(email)+len(http):,}')

In [ ]:
print('📂 Loading LDAP organisational data...\n')

ldap_files = sorted([
    f for f in os.listdir(os.path.join(CERT_DATA_PATH, 'LDAP'))
    if f.endswith('.csv')
])
print(f'   Found {len(ldap_files)} LDAP snapshot files: {ldap_files}\n')

ldap_frames = []
for lf in ldap_files:
    tmp = pd.read_csv(os.path.join(CERT_DATA_PATH, 'LDAP', lf))
    ldap_frames.append(tmp)

ldap_raw = pd.concat(ldap_frames, ignore_index=True)
ldap_raw.columns = ldap_raw.columns.str.lower().str.replace(' ', '_')

if 'user_id' not in ldap_raw.columns and 'employee_name' in ldap_raw.columns:
    ldap_raw.rename(columns={'employee_name': 'user_id'}, inplace=True)

sort_col = 'employee_id' if 'employee_id' in ldap_raw.columns else ldap_raw.columns[0]
keep_cols = [c for c in ['user_id', 'role', 'department', 'team', 'supervisor'] if c in ldap_raw.columns]

ldap = (
    ldap_raw
    .sort_values(sort_col)
    .drop_duplicates(subset=['user_id'], keep='last')
    [keep_cols]
    .reset_index(drop=True)
)

print(f'   ✅ LDAP enrichment table: {len(ldap):,} unique users')
print(f'   Roles found    : {ldap["role"].nunique()} unique')
print(f'   Departments    : {ldap["department"].nunique()} unique')
ldap.head(3)

In [ ]:
def enrich_with_ldap(df: pd.DataFrame, ldap: pd.DataFrame) -> pd.DataFrame:
    """Left-join LDAP role/department onto any event dataframe."""
    return df.merge(ldap, on='user_id', how='left')

logon  = enrich_with_ldap(logon,  ldap)
device = enrich_with_ldap(device, ldap)
files  = enrich_with_ldap(files,  ldap)
email  = enrich_with_ldap(email,  ldap)
http   = enrich_with_ldap(http,   ldap)

for name, df in [('logon', logon), ('device', device),
                 ('files', files), ('email', email), ('http', http)]:
    df['date_only']      = df['date'].dt.normalize()
    df['hour']           = df['date'].dt.hour
    df['weekday']        = df['date'].dt.dayofweek
    df['is_weekend']     = df['weekday'] >= 5
    df['is_after_hours'] = ~df['hour'].between(8, 18)

print('✅ LDAP enrichment + time columns applied to all event logs.')
print('\nSample enriched logon record:')
logon[['user_id', 'date', 'role', 'department', 'hour', 'is_after_hours', 'is_weekend']].head(3)

---
## 1.2 Dataset Overview

Quick sanity checks — unique users, date range, and event counts across all five log sources.

In [ ]:
all_users = (
    set(logon['user_id']) | set(files['user_id']) |
    set(device['user_id']) | set(email['user_id']) | set(http['user_id'])
)

date_min = min(df['date'].min() for df in [logon, files, device, email, http])
date_max = max(df['date'].max() for df in [logon, files, device, email, http])

summary = pd.DataFrame({
    'Source'      : ['logon', 'device', 'files', 'email', 'http'],
    'Rows'        : [len(logon), len(device), len(files), len(email), len(http)],
    'Unique Users': [
        logon['user_id'].nunique(),  device['user_id'].nunique(),
        files['user_id'].nunique(),  email['user_id'].nunique(),
        http['user_id'].nunique()
    ],
    'Date Min': [df['date'].min().date() for df in [logon, device, files, email, http]],
    'Date Max': [df['date'].max().date() for df in [logon, device, files, email, http]],
})

print('=' * 60)
print(f'  Total unique users : {len(all_users):,}')
print(f'  Dataset date range : {date_min.date()} → {date_max.date()}')
print(f'  Total span         : {(date_max - date_min).days} days')
print(f'  Total events       : {summary["Rows"].sum():,}')
print('=' * 60)
summary

---
## 1.3 Exploratory Data Analysis

We explore the behavioral landscape across all users before building features. Key questions:
- When are users most active?
- Which users have unusually high activity volumes?
- What does after-hours activity look like across roles?

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Daily Event Volume by Source — CERT r6.2',
             fontsize=14, fontweight='bold', y=1.01)

sources = [
    (logon,  'Logon Events',  '#1E90FF'),
    (device, 'Device Events', '#FF6B35'),
    (files,  'File Events',   '#00C853'),
    (email,  'Email Events',  '#FFD700'),
    (http,   'HTTP Events',   '#DA70D6'),
]

for ax, (df, label, color) in zip(axes, sources):
    daily = df.groupby('date_only').size().reset_index(name='count')
    ax.fill_between(daily['date_only'], daily['count'], alpha=0.4, color=color)
    ax.plot(daily['date_only'], daily['count'], color=color, linewidth=1.2)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.tight_layout()
plt.savefig('../data/eda_event_volume.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Event volume timeline saved.')

In [ ]:
hourly_data = pd.concat([
    logon[['hour', 'weekday']].assign(source='logon'),
    files[['hour', 'weekday']].assign(source='files'),
    http[['hour', 'weekday']].assign(source='http'),
])

pivot = (
    hourly_data
    .groupby(['weekday', 'hour'])
    .size()
    .reset_index(name='count')
    .pivot(index='weekday', columns='hour', values='count')
    .fillna(0)
)

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

plt.figure(figsize=(16, 4))
sns.heatmap(
    pivot,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='white',
    yticklabels=day_labels,
    cbar_kws={'label': 'Event Count'},
)
plt.title('Activity Heatmap — Hour of Day × Day of Week (All Users)',
          fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.savefig('../data/eda_hourly_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Hourly heatmap saved.')
print('\n💡 Observation: Peak activity is 09:00–17:00 Mon–Fri.')
print('   After-hours events are rare for most users — making them strong anomaly signals.')

In [ ]:
top_file_users = (
    files.groupby('user_id')
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='file_events')
)
top_file_users = top_file_users.merge(
    ldap[['user_id', 'role', 'department']], on='user_id', how='left'
)

fig = px.bar(
    top_file_users,
    x='file_events',
    y='user_id',
    color='department',
    orientation='h',
    title='Top 20 Users by Total File Access Events',
    labels={'file_events': 'Total File Events', 'user_id': 'User ID'},
    height=500,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=14,
    yaxis={'categoryorder': 'total ascending'},
)
fig.show()
print('✅ Top file access users plotted.')

In [ ]:
logon_role = (
    logon.groupby(['role', 'is_after_hours'])
    .size()
    .reset_index(name='count')
)
logon_role['pct'] = logon_role.groupby('role')['count'].transform(
    lambda x: x / x.sum() * 100
)
after_hours_role = (
    logon_role[logon_role['is_after_hours'] == True]
    .copy()
    .sort_values('pct', ascending=False)
)

fig = px.bar(
    after_hours_role,
    x='role',
    y='pct',
    title='After-Hours Login Rate by Role (%)',
    labels={'pct': 'After-Hours Login %', 'role': 'Role'},
    color='pct',
    color_continuous_scale='Reds',
    height=400,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=14,
    xaxis_tickangle=-30,
)
fig.show()
print('✅ After-hours activity by role plotted.')
print('\n💡 Observation: Roles with legitimately high after-hours rates')
print('   (e.g., IT Admin) need role-adjusted baselines to avoid false positives.')

In [ ]:
usb_by_user = (
    device.groupby('user_id')
    .size()
    .reset_index(name='usb_events')
    .sort_values('usb_events', ascending=False)
)

fig = px.histogram(
    usb_by_user,
    x='usb_events',
    nbins=50,
    title='Distribution of USB Events per User (Full Dataset)',
    labels={'usb_events': 'Total USB Events', 'count': 'Number of Users'},
    color_discrete_sequence=['#FF6B35'],
    height=350,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=14,
)
fig.show()

print(f'\n💡 Most users have very few USB events.')
print(f'   Users with 0 USB events  : {len(all_users) - len(usb_by_user):,}')
print(f'   Users with >10 USB events: {(usb_by_user["usb_events"] > 10).sum():,}')
print(f'   → First-ever USB event is a HIGH-value anomaly signal.')

---
## 1.4 Feature Engineering

We transform raw event logs into a **per-user per-day behavioral feature matrix** — the "fingerprint" of each user's activity.

Each row = one user on one day = ~40 behavioral features.

### Feature Groups
| Group | Features | Source |
|-------|----------|--------|
| 🕐 Temporal | After-hours ratio, session duration, weekend activity | logon |
| 📁 File Access | Volume, sensitivity ratio, delete ratio, bulk flag | file |
| 💾 Device | USB event count, bytes written, new device flag | device |
| 📧 Email | External ratio, attachment count, bulk send flag | email |
| 🌐 Web | Job site visits, cloud storage visits, unique domains | http |
| 📈 Velocity | 7-day rolling delta vs. 30-day personal baseline | all |
| 👥 Peer Cohort | Z-score deviation from role-cohort mean | all + LDAP |

In [ ]:
print('🔧 Building daily user-level aggregation base...')

all_user_dates = pd.concat([
    logon[['user_id',  'date_only']],
    device[['user_id', 'date_only']],
    files[['user_id',  'date_only']],
    email[['user_id',  'date_only']],
    http[['user_id',   'date_only']],
]).drop_duplicates()

base = all_user_dates.merge(
    ldap[['user_id', 'role', 'department']], on='user_id', how='left'
)
base = base.sort_values(['user_id', 'date_only']).reset_index(drop=True)

print(f'   ✅ Base frame: {len(base):,} user-day rows')
print(f'      Unique users : {base["user_id"].nunique():,}')
print(f'      Unique days  : {base["date_only"].nunique():,}')
base.head(3)

In [ ]:
print('🕐 Computing temporal features...')

pc_col = 'pc' if 'pc' in logon.columns else 'user_id'

logon_daily = (
    logon
    .groupby(['user_id', 'date_only'])
    .agg(
        login_count        = ('date', 'count'),
        after_hours_logins = ('is_after_hours', 'sum'),
        weekend_logins     = ('is_weekend', 'sum'),
        first_login_hour   = ('hour', 'min'),
        last_login_hour    = ('hour', 'max'),
        unique_pcs         = (pc_col, 'nunique'),
    )
    .reset_index()
)

logon_daily['after_hours_ratio'] = (
    logon_daily['after_hours_logins'] / logon_daily['login_count'].clip(lower=1)
)
logon_daily['session_span_hours'] = (
    logon_daily['last_login_hour'] - logon_daily['first_login_hour']
).clip(lower=0)

print(f'   ✅ Temporal features: {logon_daily.shape[1]-2} features '
      f'for {logon_daily["user_id"].nunique():,} users')

In [ ]:
print('📁 Computing file access features...')

def is_sensitive(path) -> bool:
    if not isinstance(path, str):
        return False
    path_lower = path.lower()
    return any(kw in path_lower for kw in SENSITIVE_KEYWORDS)

fname_col = 'filename' if 'filename' in files.columns else \
            'content'  if 'content'  in files.columns else None

files['is_sensitive'] = (
    files[fname_col].apply(is_sensitive) if fname_col
    else pd.Series(False, index=files.index)
)

if 'activity' in files.columns:
    files['is_delete'] = files['activity'].str.lower().str.contains('delete|remove', na=False)
    files['is_copy']   = files['activity'].str.lower().str.contains('copy|move',     na=False)
else:
    files['is_delete'] = False
    files['is_copy']   = False

dir_agg = (
    (lambda x: x.dropna().apply(lambda p: os.path.dirname(p)).nunique())
    if fname_col else (lambda x: x.count())
)

files_daily = (
    files
    .groupby(['user_id', 'date_only'])
    .agg(
        files_accessed     = ('user_id',      'count'),
        sensitive_files    = ('is_sensitive',  'sum'),
        delete_events      = ('is_delete',     'sum'),
        copy_events        = ('is_copy',       'sum'),
        unique_directories = (fname_col if fname_col else 'user_id', dir_agg),
    )
    .reset_index()
)

files_daily['sensitive_file_ratio'] = (
    files_daily['sensitive_files'] / files_daily['files_accessed'].clip(lower=1)
)
files_daily['delete_ratio'] = (
    files_daily['delete_events'] / files_daily['files_accessed'].clip(lower=1)
)
files_daily['bulk_download_flag'] = (files_daily['files_accessed'] > 50).astype(int)

print(f'   ✅ File features: {files_daily.shape[1]-2} features '
      f'for {files_daily["user_id"].nunique():,} users')

In [ ]:
print('💾 Computing device/USB features...')

bytes_col = next(
    (c for c in device.columns if 'byte' in c.lower() or 'size' in c.lower()),
    None
)

agg_dict = {
    'usb_event_count': ('user_id', 'count'),
    'after_hours_usb': ('is_after_hours', 'sum'),
}
if bytes_col:
    agg_dict['usb_bytes_written'] = (bytes_col, 'sum')

device_daily = (
    device
    .groupby(['user_id', 'date_only'])
    .agg(**agg_dict)
    .reset_index()
)

if 'usb_bytes_written' not in device_daily.columns:
    device_daily['usb_bytes_written'] = 0.0

device_daily['usb_bytes_gb'] = device_daily['usb_bytes_written'] / 1e9

first_usb = (
    device.groupby('user_id')['date_only']
    .min()
    .reset_index()
    .rename(columns={'date_only': 'first_usb_date'})
)
device_daily = device_daily.merge(first_usb, on='user_id', how='left')
device_daily['new_device_flag'] = (
    device_daily['date_only'] == device_daily['first_usb_date']
).astype(int)
device_daily.drop(columns=['first_usb_date'], inplace=True)

print(f'   ✅ Device features: {device_daily.shape[1]-2} features '
      f'for {device_daily["user_id"].nunique():,} users')

In [ ]:
print('📧 Computing email features...')

if 'to' in email.columns:
    email['is_external'] = email['to'].str.contains(
        r'@(?!dtaa\.com|company\.com)', na=False, regex=True
    )
elif 'domain' in email.columns:
    email['is_external'] = ~email['domain'].str.contains(
        'dtaa|internal|company', na=False
    )
else:
    email['is_external'] = False

att_col = next(
    (c for c in email.columns if 'attach' in c.lower() or 'size' in c.lower()),
    None
)
email['has_attachment'] = (
    email[att_col].fillna(0).astype(float) > 0 if att_col
    else pd.Series(False, index=email.index)
)

email_daily = (
    email
    .groupby(['user_id', 'date_only'])
    .agg(
        email_count             = ('user_id',         'count'),
        external_emails         = ('is_external',     'sum'),
        emails_with_attachments = ('has_attachment',  'sum'),
    )
    .reset_index()
)

email_daily['external_email_ratio'] = (
    email_daily['external_emails'] / email_daily['email_count'].clip(lower=1)
)
email_daily['bulk_send_flag'] = (email_daily['email_count'] > 30).astype(int)

print(f'   ✅ Email features: {email_daily.shape[1]-2} features '
      f'for {email_daily["user_id"].nunique():,} users')

In [ ]:
print('🌐 Computing web/HTTP features...')

url_col = next(
    (c for c in http.columns if 'url' in c.lower() or 'domain' in c.lower()),
    None
)

def classify_url(url) -> dict:
    if not isinstance(url, str):
        return {'is_job_site': 0, 'is_cloud_storage': 0, 'is_webmail': 0}
    url_lower = url.lower()
    return {
        'is_job_site'      : int(any(d in url_lower for d in JOB_SITE_DOMAINS)),
        'is_cloud_storage' : int(any(d in url_lower for d in CLOUD_STORAGE_DOMAINS)),
        'is_webmail'       : int(any(d in url_lower for d in WEBMAIL_DOMAINS)),
    }

if url_col:
    url_classes = http[url_col].apply(classify_url).apply(pd.Series)
    http = pd.concat([http, url_classes], axis=1)
else:
    http['is_job_site'] = 0
    http['is_cloud_storage'] = 0
    http['is_webmail'] = 0

def extract_domain(u):
    if not isinstance(u, str):
        return 'unknown'
    return u.split('/')[2] if '://' in u else u

domain_agg = (
    (lambda x: x.dropna().apply(extract_domain).nunique())
    if url_col else (lambda x: x.count())
)

http_daily = (
    http
    .groupby(['user_id', 'date_only'])
    .agg(
        http_requests        = ('user_id',           'count'),
        job_site_visits      = ('is_job_site',        'sum'),
        cloud_storage_visits = ('is_cloud_storage',   'sum'),
        webmail_visits       = ('is_webmail',         'sum'),
        unique_domains       = (url_col if url_col else 'user_id', domain_agg),
    )
    .reset_index()
)

print(f'   ✅ Web features: {http_daily.shape[1]-2} features '
      f'for {http_daily["user_id"].nunique():,} users')

In [ ]:
print('🔗 Merging all feature groups into unified daily matrix...')

feature_matrix = base.copy()

for daily_df, name in [
    (logon_daily,  'temporal'),
    (files_daily,  'file'),
    (device_daily, 'device'),
    (email_daily,  'email'),
    (http_daily,   'web'),
]:
    feature_matrix = feature_matrix.merge(
        daily_df, on=['user_id', 'date_only'], how='left'
    )
    print(f'   ✅ Merged {name:10s} features → shape: {feature_matrix.shape}')

numeric_cols = feature_matrix.select_dtypes(include=[np.number]).columns
feature_matrix[numeric_cols] = feature_matrix[numeric_cols].fillna(0)

print(f'\n📐 Feature matrix shape: {feature_matrix.shape}')
print(f'   {feature_matrix["user_id"].nunique():,} users × '
      f'{feature_matrix["date_only"].nunique():,} days × '
      f'{len(numeric_cols)} numeric features')

In [ ]:
print('📈 Computing velocity features (rolling baseline comparison)...')

VELOCITY_FEATURES = [
    'files_accessed', 'email_count', 'usb_event_count',
    'after_hours_ratio', 'job_site_visits', 'cloud_storage_visits',
    'external_email_ratio', 'sensitive_file_ratio',
]
velocity_features_present = [
    f for f in VELOCITY_FEATURES if f in feature_matrix.columns
]

feature_matrix = feature_matrix.sort_values(['user_id', 'date_only'])

for feat in tqdm(velocity_features_present, desc='Velocity features'):
    feature_matrix[f'{feat}_baseline30'] = (
        feature_matrix.groupby('user_id')[feat]
        .transform(lambda x: x.rolling(BASELINE_WINDOW_DAYS, min_periods=5).mean().shift(1))
    )
    feature_matrix[f'{feat}_roll7'] = (
        feature_matrix.groupby('user_id')[feat]
        .transform(lambda x: x.rolling(7, min_periods=1).mean())
    )
    feature_matrix[f'{feat}_velocity'] = (
        (
            feature_matrix[f'{feat}_roll7'] -
            feature_matrix[f'{feat}_baseline30']
        ) / feature_matrix[f'{feat}_baseline30'].abs().clip(lower=0.01)
    ).clip(-10, 10)

print(f'\n   ✅ Velocity features added: {len(velocity_features_present) * 3} new columns')

In [ ]:
print('👥 Computing peer cohort deviation features...')

COHORT_FEATURES = [
    'files_accessed', 'email_count', 'usb_event_count',
    'after_hours_ratio', 'external_email_ratio',
]
cohort_features_present = [
    f for f in COHORT_FEATURES if f in feature_matrix.columns
]

for feat in tqdm(cohort_features_present, desc='Cohort features'):
    cohort_stats = (
        feature_matrix.groupby(['role', 'date_only'])[feat]
        .agg(cohort_mean='mean', cohort_std='std')
        .reset_index()
    )
    cohort_stats['cohort_std'] = cohort_stats['cohort_std'].fillna(1).clip(lower=0.01)

    feature_matrix = feature_matrix.merge(
        cohort_stats, on=['role', 'date_only'], how='left'
    )
    feature_matrix[f'{feat}_cohort_zscore'] = (
        (feature_matrix[feat] - feature_matrix['cohort_mean']) /
        feature_matrix['cohort_std']
    ).clip(-5, 5)
    feature_matrix.drop(columns=['cohort_mean', 'cohort_std'], inplace=True)

print(f'\n   ✅ Cohort z-score features added: {len(cohort_features_present)} new columns')
print(f'\n📐 Final feature matrix shape: {feature_matrix.shape}')

In [ ]:
numeric_cols = feature_matrix.select_dtypes(include=[np.number]).columns.tolist()

print('=' * 60)
print('  FEATURE MATRIX SUMMARY')
print('=' * 60)
print(f'  Total rows (user-days)  : {len(feature_matrix):,}')
print(f'  Unique users            : {feature_matrix["user_id"].nunique():,}')
print(f'  Unique days             : {feature_matrix["date_only"].nunique():,}')
print(f'  Total numeric features  : {len(numeric_cols)}')
print(f'  Missing values          : {feature_matrix[numeric_cols].isna().sum().sum():,}')
print('=' * 60)

groups = {
    'Temporal'   : [c for c in numeric_cols if any(k in c for k in
                    ['login', 'after_hours', 'session', 'weekend', 'hour'])],
    'File Access': [c for c in numeric_cols if any(k in c for k in
                    ['file', 'sensitive', 'delete', 'copy', 'bulk', 'dir'])],
    'Device/USB' : [c for c in numeric_cols if 'usb' in c or 'device' in c],
    'Email'      : [c for c in numeric_cols if 'email' in c or 'attach' in c],
    'Web'        : [c for c in numeric_cols if any(k in c for k in
                    ['http', 'job_site', 'cloud', 'webmail', 'domain'])],
    'Velocity'   : [c for c in numeric_cols if 'velocity' in c],
    'Cohort'     : [c for c in numeric_cols if 'cohort' in c],
}

print('\n  Feature breakdown by group:')
for group, cols in groups.items():
    print(f'  {group:<15}: {len(cols):>3} features')

In [ ]:
sample_users = (
    feature_matrix.groupby('user_id')['files_accessed']
    .sum()
    .sort_values(ascending=False)
    .head(30)
    .index.tolist()
)

viz_features = [
    'files_accessed', 'after_hours_ratio', 'usb_event_count',
    'external_email_ratio', 'job_site_visits', 'sensitive_file_ratio',
    'email_count', 'cloud_storage_visits',
]
viz_features_present = [f for f in viz_features if f in feature_matrix.columns]

user_avg = (
    feature_matrix[feature_matrix['user_id'].isin(sample_users)]
    .groupby('user_id')[viz_features_present]
    .mean()
)
user_avg_norm = (
    (user_avg - user_avg.min()) / (user_avg.max() - user_avg.min() + 1e-9)
)

fig = px.imshow(
    user_avg_norm,
    title='Behavioral Fingerprint Heatmap — Top 30 Users (Normalised)',
    labels={'x': 'Feature', 'y': 'User ID', 'color': 'Normalised Value'},
    color_continuous_scale='RdYlGn_r',
    aspect='auto',
    height=600,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=13,
)
fig.show()
print('✅ Behavioral fingerprint heatmap rendered.')
print('💡 Red = high activity. Users with many red cells across features')
print('   are candidates for deeper anomaly investigation.')

---
## 1.5 Save Feature Matrix

In [ ]:
os.makedirs('../data', exist_ok=True)

output_parquet = '../data/feature_matrix.parquet'
output_csv     = '../data/feature_matrix.csv'

feature_matrix.to_parquet(output_parquet, index=False)
feature_matrix.to_csv(output_csv, index=False)

print('=' * 60)
print('  ✅ Feature matrix saved!')
print(f'     Parquet : {output_parquet}')
print(f'     CSV     : {output_csv}')
print(f'     Shape   : {feature_matrix.shape}')
print(f'     Size    : {os.path.getsize(output_parquet) / 1e6:.1f} MB (parquet)')
print('=' * 60)
print('\n🎯 Notebook 01 Complete!')
print('   → Open 02_ml_anomaly_detection.ipynb to train the ML models.')

---
## ✅ Notebook 01 — Summary

### What We Built
| Step | Output |
|------|--------|
| Loaded 5 CERT event log CSVs | ~16M raw events |
| Enriched with LDAP org data | Role + department per user |
| Explored behavioral patterns | 5 EDA visualisations |
| Engineered 7 feature groups | ~40+ behavioral features |
| Built velocity baselines | 7-day vs. 30-day rolling delta |
| Built peer cohort features | Z-score vs. role group |
| Saved feature matrix | `../data/feature_matrix.parquet` |

### Key Observations
- 🕐 After-hours activity is rare for most roles → strong anomaly signal
- 💾 USB events are near-zero for most users → first event is highly suspicious
- 📁 Sensitive file access is concentrated in a small subset of users
- 👥 Role-adjusted baselines are essential to avoid false positives

### Next Step
**Notebook 02** — Train Isolation Forest, Autoencoder, and LSTM on this feature matrix to generate per-user anomaly scores.